# Generic Performance Review Notebook

This notebook demonstrates a simplified workflow for:
- loading operational data
- cleaning timestamped records
- deriving reporting periods and shifts
- aggregating planned versus actual performance
- exporting a summary table

All site-specific, internal, and proprietary labels have been replaced with generic names.

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import openpyxl
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment

In [ ]:
raw_data = pd.read_csv("path/to/input_data.csv")

In [ ]:
raw_data.head(10)

In [ ]:
trimmed_data = raw_data.drop([0, 1, 2, 3, 4]).reset_index()
trimmed_data.head(10)

In [ ]:
cleaned_data = trimmed_data.drop(["index"], axis=1)
cleaned_data.head(20)

In [ ]:
cleaned_data.shape

In [ ]:
cleaned_data.describe(include="all")

In [ ]:
cleaned_data.info()

In [ ]:
cleaned_data = cleaned_data.drop(
    ["override_column_1", "override_column_2"],
    axis=1,
    errors="ignore"
)

In [ ]:
cleaned_data.info()

In [ ]:
cleaned_data["timestamp"] = pd.to_datetime(cleaned_data["timestamp"])
cleaned_data.info()

In [ ]:
cleaned_data["date_only"] = pd.to_datetime(cleaned_data["timestamp"]).dt.strftime("%Y/%m/%d")
cleaned_data["time_only"] = pd.to_datetime(cleaned_data["timestamp"]).dt.strftime("%H:%M")

cleaned_data.head(250)

In [ ]:
def assign_period(time_str):
    time_only = time_str.split() if " " in time_str else time_str
    hour, minute = map(int, time_only.split(":"))

    if (hour == 8 and minute >= 0) or (8 < hour < 10) or (hour == 10 and minute < 35):
        return "P1"
    elif (hour == 11 and minute >= 0) or (11 < hour < 14) or (hour == 14 and minute < 35):
        return "P2"
    elif (hour == 15 and minute >= 0) or (15 < hour < 18) or (hour == 18 and minute < 35):
        return "P3"
    elif (hour == 18 and minute >= 35) or (18 < hour < 20) or (hour == 20 and minute < 5):
        return "P4"
    elif (hour == 20 and minute >= 30) or (20 < hour < 24) or (0 <= hour < 1) or (hour == 1 and minute < 50):
        return "P5"
    elif (hour == 2 and minute >= 15) or (2 < hour < 5) or (hour == 5 and minute < 5):
        return "P6"
    else:
        return "null"

cleaned_data["Period"] = cleaned_data["time_only"].apply(assign_period)

In [ ]:
cleaned_data.head(250)

In [ ]:
cleaned_data.info()

In [ ]:
def assign_shift(time_str):
    time_only = time_str.split() if " " in time_str else time_str
    hour, minute = map(int, time_only.split(":"))
    decimal_time = hour + minute / 60

    if 8.0 <= decimal_time < 18.5:
        return "Day"
    elif decimal_time >= 19.0 or decimal_time < 5.0:
        return "Night"
    else:
        return "null"

cleaned_data["Shift"] = cleaned_data["time_only"].apply(assign_shift)

In [ ]:
cleaned_data.head(300)

In [ ]:
cleaned_data_sorted = cleaned_data.sort_values(by=["Period", "Shift"])
cleaned_data_sorted.head(100)

In [ ]:
cleaned_data.info()

In [ ]:
cleaned_data["planned_value"] = pd.to_numeric(
    cleaned_data["planned_value"],
    errors="coerce"
)

cleaned_data["actual_value"] = pd.to_numeric(
    cleaned_data["actual_value"],
    errors="coerce"
)

filtered_data = cleaned_data.dropna(subset=[
    "Period",
    "Shift",
    "planned_value",
    "actual_value"
])

filtered_data = filtered_data[
    (filtered_data["Period"] != "null") &
    (filtered_data["Shift"] != "null")
]

summary_table = filtered_data.groupby(["Period", "Shift"]).agg({
    "planned_value": "mean",
    "actual_value": "mean"
}).reset_index()

summary_table.columns = [
    "Period",
    "Shift",
    "Planned Average",
    "Actual Average"
]

summary_table["Planned Average"] = summary_table["Planned Average"].round(2)
summary_table["Actual Average"] = summary_table["Actual Average"].round(2)

summary_table["Variance %"] = (
    (summary_table["Actual Average"] - summary_table["Planned Average"])
    / summary_table["Planned Average"]
) * 100
summary_table["Variance %"] = summary_table["Variance %"].round(2)

summary_table["Status"] = np.where(
    abs(summary_table["Variance %"]) <= 5,
    "Aligned",
    "Not Aligned"
)

summary_table = summary_table[
    ["Period", "Shift", "Variance %", "Planned Average", "Actual Average", "Status"]
]

summary_table = summary_table.sort_values(by=["Period", "Shift"])
summary_table.head()

In [ ]:
output_file = "performance_summary.xlsx"
summary_table.to_excel(output_file, index=False, sheet_name="Summary")

print(f"Summary table exported successfully to: {output_file}")